# HD-Referenzsätze – Worthäufigkeit & Korpusanalyse

**Forschungsfrage:**  
*«Wie gut lässt sich der dialektspezifische Wortschatz des Ostschweizer Dialekts automatisch hochdeutschen Äquivalenten zuordnen – gemessen an Wörtern mittlerer Häufigkeit?»*

Dieses Notebook analysiert die hochdeutschen Referenzsätze des Ostschweizer Teilkorpus hinsichtlich Worthäufigkeitsverteilung, Vokabular-Eigenschaften und weiterer Points of Interest (POI), die für die Forschungsfrage relevant sind.

In [ ]:
import re
from collections import Counter

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from wordcloud import WordCloud

plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

df = pd.read_csv(DATA_DIR / "transcriptions_clean.csv")
df_ost = df[df["dialect_region"] == "Ostschweiz"].copy().reset_index(drop=True)

# ── Tokenisierung ─────────────────────────────────────────────────────────────
def tokenize(text):
    """Kleinbuchstaben, nur Buchstaben (inkl. Umlaute)."""
    return re.findall(r"[a-zäöüß]+", str(text).lower())

df_ost["tokens"] = df_ost["sentence"].apply(tokenize)
all_tokens = [tok for sent in df_ost["tokens"] for tok in sent]
freq = Counter(all_tokens)

print(f"Sätze:          {len(df_ost):,}")
print(f"Tokens total:   {len(all_tokens):,}")
print(f"Types (Vocab):  {len(freq):,}")
print(f"Type-Token-Ratio (TTR): {len(freq)/len(all_tokens):.4f}")

## 1. Top-Wörter (inkl. Funktionswörter)

In [ ]:
top_n = 30
top_words, top_counts = zip(*freq.most_common(top_n))

# IPA-Tokenisierung (Wort-Ebene, ipa_audio)
df_ost["ipa_tokens"] = df_ost["ipa_audio"].apply(lambda x: str(x).strip().split())
all_ipa = [tok for sent in df_ost["ipa_tokens"] for tok in sent]
ipa_freq = Counter(all_ipa)
top_ipa = ipa_freq.most_common(top_n)
ipa_words, ipa_counts = zip(*top_ipa)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Häufigste Wörter
bars = axes[0].barh(top_words[::-1], top_counts[::-1], color="#4472C4")
axes[0].set_xlabel("Häufigkeit")
axes[0].set_title(f"Top {top_n} häufigste Wörter")
axes[0].bar_label(bars, padding=3, fontsize=8)

# Häufigste IPA-Token
bars2 = axes[1].barh(ipa_words[::-1], ipa_counts[::-1], color="#E67E22")
axes[1].set_xlabel("Häufigkeit")
axes[1].set_title(f"Top {top_n} häufigste IPA-Token (ipa_audio)")
axes[1].bar_label(bars2, padding=3, fontsize=8)

plt.suptitle("Worthäufigkeit – Rohkorpus vs. IPA (Ostschweiz)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"IPA-Token total:  {len(all_ipa):,}")
print(f"IPA-Types:        {len(ipa_freq):,}")

In [ ]:
# Liniendiagramm: Anzahl Wörter die genau 1x, 2x, ... 20x vorkommen
max_freq = 20
x = list(range(1, max_freq + 1))
y = [sum(1 for c in freq.values() if c == n) for n in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, marker="o", color="#4472C4", linewidth=2, markersize=6)
ax.fill_between(x, y, alpha=0.15, color="#4472C4")

for xi, yi in zip(x, y):
    ax.annotate(str(yi), xy=(xi, yi), xytext=(0, 6), textcoords="offset points",
                ha="center", fontsize=8, color="#2c3e50")

ax.set_xticks(x)
ax.set_xlabel("Anzahl Vorkommen im Korpus")
ax.set_ylabel("Anzahl Wörter (Types)")
ax.set_title("Wie viele Wörter kommen genau 1×, 2×, … 20× vor?")
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# HD Anzahl Wörter pro Satz vs Anzahl IPA Token pro Satz für alle Zeiten
df_tenses = pd.read_csv(DATA_DIR / "transcriptions_tenses.csv")
df_ost_t = df_tenses[df_tenses["dialect_region"] == "Ostschweiz"].copy()

df_ost_t["hd_word_count"]  = df_ost_t["sentence"].apply(lambda x: len(str(x).split()))
df_ost_t["ipa_token_count"] = df_ost_t["ipa_audio"].apply(lambda x: len(str(x).strip().split()))

tense_order = ["Präsens", "Perfekt", "Plusquamperfekt", "Präteritum", "Futur"]
tense_order = [t for t in tense_order if t in df_ost_t["tense"].unique()]

hd_means  = [df_ost_t[df_ost_t["tense"] == t]["hd_word_count"].mean()  for t in tense_order]
ipa_means = [df_ost_t[df_ost_t["tense"] == t]["ipa_token_count"].mean() for t in tense_order]
hd_stds   = [df_ost_t[df_ost_t["tense"] == t]["hd_word_count"].std()   for t in tense_order]
ipa_stds  = [df_ost_t[df_ost_t["tense"] == t]["ipa_token_count"].std()  for t in tense_order]

x = np.arange(len(tense_order))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, hd_means, width, yerr=hd_stds, label="HD Wörter",  color="#4472C4", capsize=4, alpha=0.85)
bars2 = ax.bar(x + width/2, ipa_means, width, yerr=ipa_stds, label="IPA Token", color="#E67E22", capsize=4, alpha=0.85)

ax.set_xlabel("Grammatikalische Zeit")
ax.set_ylabel("Ø Anzahl pro Satz")
ax.set_title("HD-Wörter vs. IPA-Token pro Satz – nach Zeit (Ostschweiz)")
ax.set_xticks(x)
ax.set_xticklabels(tense_order)
ax.legend()
ax.bar_label(bars1, fmt="%.1f", padding=3, fontsize=8)
ax.bar_label(bars2, fmt="%.1f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

print(df_ost_t.groupby("tense")[["hd_word_count", "ipa_token_count"]].agg(["mean", "std"]).round(2).to_string())